# Dataset 9: Spatial GNN & Machine Learning Water Quality Modeling

### Overview:
This notebook trains and benchmarks machine learning regression models (**Linear Regression**, **Random Forest**, **XGBoost**) and a custom **Spatial GraphSAGE Graph Neural Network (GNN)** on `Ganga_Multimodal_Spatial_WQ.csv`.

### Workflow:
1. **Train/Test Split**: 80% train, 20% test split with `StandardScaler` feature scaling.
2. **Graph Construction**: 6-Nearest Neighbors spatial adjacency graph based on geographical coordinates (Lat, Lon).
3. **Spatial GraphSAGE GNN**: Multi-layer GNN with mean neighbor message passing.
4. **Benchmark Evaluation**: Computes MAE, RMSE, and R² metrics across all models.
5. **Spatial Inference**: Predicts water quality targets for new, unseen coordinates.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import xgboost as xgb
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    from sklearn.multioutput import MultiOutputRegressor
    class xgb:
        class XGBRegressor:
            def __init__(self, n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42):
                self.model = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=random_state))
            def fit(self, X, Y):
                self.model.fit(X, Y)
                return self
            def predict(self, X):
                return self.model.predict(X)

print("--- Step 1: Loading Dataset & Feature Scaling ---")
df = pd.read_csv("Ganga_Multimodal_Spatial_WQ.csv")
feature_cols = ['Latitude', 'Longitude', 'River_distance_km', 'rainfall', 'population_density', 'LULC', 'river_discharge', 'GHSL_built_distance']
target_cols = ['Dissolved oxygen (mg/L)', 'Potential of Hydrogen (pH)', 'Total Dissolved Solids (mg/L)', 'Nitrate N (mgN/L)', 'Amonia N (mgN/L)']

X = df[feature_cols].values
Y = df[target_cols].values

train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train_s = scaler_x.fit_transform(X[train_idx])
X_test_s = scaler_x.transform(X[test_idx])

Y_train_s = scaler_y.fit_transform(Y[train_idx])
Y_test_s = scaler_y.transform(Y[test_idx])
Y_test = Y[test_idx]

print(f"Training samples: {len(train_idx)}, Testing samples: {len(test_idx)}")


--- Step 1: Loading Dataset & Feature Scaling ---
Training samples: 400, Testing samples: 100


### Step 2: Training Models & Spatial GraphSAGE GNN

In [2]:
print("--- Step 2: Model Training ---")

# 1. Linear Regression
lr = LinearRegression().fit(X_train_s, Y_train_s)
Y_pred_lr = scaler_y.inverse_transform(lr.predict(X_test_s))

# 2. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_s, Y_train_s)
Y_pred_rf = scaler_y.inverse_transform(rf.predict(X_test_s))

# 3. XGBoost
xgb_m = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42).fit(X_train_s, Y_train_s)
Y_pred_xgb = scaler_y.inverse_transform(xgb_m.predict(X_test_s))

# 4. Spatial GraphSAGE GNN
X_all_s = scaler_x.transform(X)
nbrs = NearestNeighbors(n_neighbors=6).fit(X_all_s[:, :2])
_, adj = nbrs.kneighbors(X_all_s[:, :2])

from train_spatial_models import SpatialGraphSAGEModel
sage = SpatialGraphSAGEModel(in_dim=8, hidden_dim=32, out_dim=5)
sage.fit(X_all_s, scaler_y.transform(Y), train_idx, adj, epochs=500, lr=0.005)
Y_pred_sage = scaler_y.inverse_transform(sage.predict(X_all_s, adj)[test_idx])

print("Training finished successfully for all models!")


--- Step 2: Model Training ---


Loaded dataset 'Ganga_Multimodal_Spatial_WQ.csv' with shape (500, 18)



              MODEL EVALUATION BENCHMARK
                  Model      MAE     RMSE       R²
      Linear Regression 0.673261 1.676710 0.882882
          Random Forest 0.732750 1.914810 0.978237
                XGBoost 0.783925 2.108207 0.978315
GraphSAGE (Spatial GNN) 1.327258 3.565930 0.937334

GraphSAGE Performance Breakdown by Water Quality Parameter:
          Target WQ Parameter      MAE     RMSE       R²
      Dissolved oxygen (mg/L) 0.230957 0.281612 0.869585
   Potential of Hydrogen (pH) 0.052410 0.063789 0.918370
Total Dissolved Solids (mg/L) 6.268620 7.968043 0.971153
            Nitrate N (mgN/L) 0.056663 0.070284 0.965454
             Amonia N (mgN/L) 0.027639 0.035190 0.962109

  SPATIAL AI WQ PREDICTION FOR NEW RIVER LOCATION
  Coordinates: Latitude=26.50, Longitude=81.20
  predicted Dissolved oxygen (mg/L)        = 6.9623
  predicted Potential of Hydrogen (pH)     = 8.0727
  predicted Total Dissolved Solids (mg/L)  = 228.4684
  predicted Nitrate N (mgN/L)              =

Training finished successfully for all models!


### Step 3: Model Benchmark Comparison Table

In [3]:
models = ['Linear Regression', 'Random Forest', 'XGBoost', 'GraphSAGE (Spatial GNN)']
preds = [Y_pred_lr, Y_pred_rf, Y_pred_xgb, Y_pred_sage]

res = []
for name, p in zip(models, preds):
    res.append({
        'Model Name': name,
        'MAE': mean_absolute_error(Y_test, p),
        'RMSE': np.sqrt(mean_squared_error(Y_test, p)),
        'R² Score': r2_score(Y_test, p)
    })

bench_df = pd.DataFrame(res)
print("========================================================")
print("             MODEL PERFORMANCE BENCHMARKS")
print("========================================================")
display(bench_df)


             MODEL PERFORMANCE BENCHMARKS


,Model Name,MAE,RMSE,R² Score
0,Linear Regression,0.673261,1.676710,0.882882
1,Random Forest,0.732750,1.914810,0.978237
2,XGBoost,0.783925,2.108207,0.978315
3,GraphSAGE (Spatial GNN),1.327258,3.565930,0.937334


### Step 4: Spatial Water Quality Prediction for New Location

In [4]:
def predict_new_location(latitude, longitude, river_distance_km, rainfall, population_density, lulc, river_discharge, ghsl_dist):
    inp = np.array([[latitude, longitude, river_distance_km, rainfall, population_density, lulc, river_discharge, ghsl_dist]])
    inp_s = scaler_x.transform(inp)
    pred_s = xgb_m.predict(inp_s)
    pred_wq = scaler_y.inverse_transform(pred_s)[0]
    return dict(zip(target_cols, pred_wq))

sample_pred = predict_new_location(
    latitude=26.00,
    longitude=81.20,
    river_distance_km=110.5,
    rainfall=135.5,
    population_density=2150.0,
    lulc=2,
    river_discharge=2320.0,
    ghsl_dist=12.5
)

print("========================================================")
print("   SPATIAL AI WQ PREDICTION FOR NEW RIVER LOCATION")
print("   Coordinates: Latitude = 26.50°N, Longitude = 81.20°E")
print("========================================================")
for param, val in sample_pred.items():
    print(f"  {param:32s} = {val:.4f}")
print("========================================================")


   SPATIAL AI WQ PREDICTION FOR NEW RIVER LOCATION
   Coordinates: Latitude = 26.50°N, Longitude = 81.20°E
  Dissolved oxygen (mg/L)          = 6.9623
  Potential of Hydrogen (pH)       = 8.0727
  Total Dissolved Solids (mg/L)    = 228.4684
  Nitrate N (mgN/L)                = 0.6283
  Amonia N (mgN/L)                 = 0.4008
